# Continual Learning — Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [ ]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method — choose one:
#   "full_ft"  — fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     — parameter-efficient LoRA adapters
#   "smf"      — frozen backbone + sparse trainable memory
#   "casm"     — versioned memory bank with contradiction-aware routing
METHOD = "full_ft"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name — checkpoints are stored under this ID
RUN_ID = "llama1b_overtrain_01"

# Periods to train — full sequence is all four in order
# Each period is one temporal snapshot of Wikipedia (~2021)
PERIODS = ["aug_sep", "sep_oct"]  # add "oct_nov", "nov_dec" for full run

# Google Drive folder — checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning"

# ---- Dataset source ----
# True  → training passages come from data/augmented/TWiki_Diffsets/<period>.csv
#          (committed in the repo; no download needed)
# False → training passages come from TWiki_Diffsets.zip
#          (must be present in data/TWiki_Diffsets.zip)
USE_AUGMENTED_DATA = True

# ---- Dataset fraction ----
# Fraction of the dataset to use for both training and evaluation.
# The fraction is applied proportionally and independently to the changed and
# unchanged probe splits, so training and evaluation always cover the same
# examples.  None means use all data; 0.1 means use 10% of each split.
DATASET_FRACTION = None  # e.g. 0.1 for 10%, or None for all data

# ---- Core hyperparameters ----
BATCH_SIZE              = 1
GRAD_ACCUM_STEPS        = 4       # effective batch size = 4
LEARNING_RATE           = 5e-4
EPOCHS_PER_PERIOD       = 7       # 7 passes over all passages
MAX_PASSAGES_PER_PERIOD = None    # use all available passages
PRECISION               = "bfloat16"
SEED                    = 42

# ---- Activation capture ----
# Records per-layer hidden-state activations after each period.
# Output saved to periods/<period>/activations.pt and activation_metadata.json
# alongside eval outputs — copy to another machine for plotting, no GPU needed.
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_NUM_SLOTS              = 8
CASM_ROUTER_HIDDEN_SIZE     = 256
CASM_TOP_K                  = 2
CASM_ROUTER_TEMPERATURE     = 1.0
CASM_SPARSITY_WEIGHT        = 0.01
CASM_OVERLAP_WEIGHT         = 0.01
CASM_BRANCH_ON_CONTRADICTION = True
CASM_MEMORY_SIZE            = 64

## Setup

In [ ]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

In [ ]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

In [ ]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required — Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

## Dataset

Training passages come from `data/augmented/TWiki_Diffsets/<period>.csv` — these are
committed directly in the repo, so no download is needed.

Evaluation probes come from `TWiki_Probes.zip` which is also bundled in the repo under
`casf_dataset_api/download_dataset_scripts/data/`.  `build_augmented_dataset` patches
the dataset loader to use that bundled copy automatically.

In [ ]:
import os

if USE_AUGMENTED_DATA:
    # Verify bundled probes zip (used for evaluation probes)
    bundled_probes = os.path.join(
        REPO_DIR, "casf_dataset_api", "download_dataset_scripts", "data", "TWiki_Probes.zip"
    )
    assert os.path.exists(bundled_probes), (
        f"TWiki_Probes.zip not found at {bundled_probes}. "
        "Make sure the repo was cloned correctly."
    )
    print(f"TWiki_Probes.zip: OK ({os.path.getsize(bundled_probes):,} bytes)")

    # Verify augmented training CSVs
    aug_dir = os.path.join(REPO_DIR, "data", "augmented", "TWiki_Diffsets")
    for period in PERIODS:
        csv_path = os.path.join(aug_dir, f"{period}.csv")
        assert os.path.exists(csv_path), f"Augmented CSV missing: {csv_path}"
        print(f"  {period}.csv: OK ({os.path.getsize(csv_path):,} bytes)")

    print("\nAll dataset files present — no download needed.")
else:
    # Verify TWiki_Diffsets.zip exists in data/
    diffsets_zip = os.path.join(REPO_DIR, "data", "TWiki_Diffsets.zip")
    probes_zip   = os.path.join(REPO_DIR, "data", "TWiki_Probes.zip")
    assert os.path.exists(diffsets_zip), f"TWiki_Diffsets.zip not found at {diffsets_zip}"
    assert os.path.exists(probes_zip),   f"TWiki_Probes.zip not found at {probes_zip}"
    print(f"TWiki_Diffsets.zip: OK ({os.path.getsize(diffsets_zip):,} bytes)")
    print(f"TWiki_Probes.zip:   OK ({os.path.getsize(probes_zip):,} bytes)")

## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # will prompt for your HuggingFace access token

## Training

In [ ]:
import os

if USE_AUGMENTED_DATA:
    aug_dir = os.path.join(REPO_DIR, "data", "augmented", "TWiki_Diffsets")
    all_ok = True
    for period in PERIODS:
        path = os.path.join(aug_dir, f"{period}.csv")
        exists = os.path.exists(path)
        size = os.path.getsize(path) if exists else 0
        status = "OK" if exists else "MISSING"
        print(f"  {period}.csv: {status} ({size:,} bytes)")
        if not exists:
            all_ok = False

    if all_ok:
        print("\nAll augmented CSVs found — training will use augmented passages.")
    else:
        raise FileNotFoundError(
            "One or more augmented CSVs are missing. "
            "Check that data/augmented/TWiki_Diffsets/<period>.csv files are committed to the repo."
        )
else:
    print("USE_AUGMENTED_DATA=False — training will use TWiki_Diffsets.zip passages.")

In [ ]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="temporal_wiki",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    dataset_fraction=DATASET_FRACTION,
    log_every_n_steps=10,
    eval_after_each_period=True,
    capture_activations=CAPTURE_ACTIVATIONS,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:               {cfg.method}")
print(f"Model:                {cfg.model_name}")
print(f"Periods:              {PERIODS}")
print(f"Batch size:           {cfg.batch_size}")
print(f"Grad accum steps:     {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:        {cfg.learning_rate}")
print(f"Epochs per period:    {cfg.epochs_per_period}")
print(f"Max passages:         {cfg.max_passages_per_period}")
print(f"Dataset fraction:     {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print(f"Capture activations:  {cfg.capture_activations}")
print(f"Checkpoint dir:       {CHECKPOINT_DIR}")

In [ ]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint — starting fresh.")

In [ ]:
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_augmented_dataset,
    build_real_dataset,
)

dataset_factory = build_augmented_dataset if USE_AUGMENTED_DATA else build_real_dataset
print(f"Dataset factory: {'augmented CSVs' if USE_AUGMENTED_DATA else 'TWiki_Diffsets.zip'}")

results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=dataset_factory,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  Eval [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

In [ ]:
# Print eval summary from the completed run (no retraining needed)
print("\n" + "="*50)
print("EVAL SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    else:
        print("  (no evaluation results)")